# Paper Energy Landscape PDFs

Post-process a saved architecture sweep directory and create clean, dense contour-only PDF energy landscapes for every saved model. Existing PNG/NPZ landscape files are left untouched; new outputs are written with a `_paper.pdf` suffix.

In [1]:
from jax import config
config.update("jax_enable_x64", True)

import json
import os
from dataclasses import replace
from pathlib import Path

import equinox as eqx
import jax
import matplotlib.pyplot as plt
import numpy as np
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "run_architectures.py").is_file():
    candidate = Path("examples/slinky/slinky_2D").resolve()
    if candidate.is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from run_architectures import (
    Dataset,
    build_architecture_registry,
    get_slinky,
    make_model_params,
    _find_saved_experiment_dirs,
    _properties_from_payload,
    _resolve_saved_data_path,
    _sweep_config_from_payload,
)
from util_energy_plots import (
    EnergyLandscapeSpec,
    _auto_limits_from_path,
    _make_fixed_strains,
    evaluate_energy_on_grid,
    extract_strain_path,
    resolve_triplet_idx,
    solve_with_strain_history,
)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 180,
    "savefig.dpi": 600,
})

## User Settings

In [2]:
# Point this at a full architecture sweep output directory.
SWEEP_OUTPUT_DIR = "arch_sweep_outputs_n11_tape_all_architectures_train_fail_on_nonconvergence_True"

# By default PDFs go into each experiment's existing energy_landscapes folder.
# Set CENTRAL_OUTPUT_DIR to a path if you want all PDFs copied into one folder.
CENTRAL_OUTPUT_DIR = None

# Which saved model snapshots to render.
INCLUDE_FINAL = True
INCLUDE_INITIAL = True

# Landscape data source and trajectories.
USE_VALID = None
# Use "all" for every trajectory, None for the saved sweep trajectory, or an int/list of ints.
TRAJ_IDXS = "all"

# Denser contours / smoother grid than the default training snapshots.
PAPER_N_GRID = 241
PAPER_N_LEVELS = 32
PAPER_CONTOUR_LINEWIDTH = 0.55
PAPER_CMAP = "viridis"

# Clean paper figure: no axes, labels, title, legend, or colorbar.
SHOW_PATH = True
PATH_COLOR = "black"
PATH_LINEWIDTH = 1.1
START_MARKER_SIZE = 18
END_MARKER_SIZE = 42
SHOW_AXES = False
SHOW_COLORBAR = False
FIGSIZE = (2.6, 2.6)
TRANSPARENT = True

# Draw numerical energy labels directly on selected contour lines.
SHOW_CONTOUR_LABELS = True
CONTOUR_LABEL_FONT_SIZE = 5.5
CONTOUR_LABEL_FORMAT = "%.1e"
CONTOUR_LABEL_EVERY = 3

# Widen automatically inferred strain limits around each trajectory.
# 1.0 reproduces the original auto range; larger values reveal more curvature.
# Optional manual limits override auto limits for every plot when not None.
STRAIN_RANGE_SCALE = 1.0
STRAIN_MIN_HALF_WIDTH = 3.5e-4
MANUAL_XLIM = None
MANUAL_YLIM = None

# Robust per-plot level limits avoid one extreme energy value flattening the contour plot.
# Set LEVEL_PERCENTILES=None to use the full finite energy range for each plot.
LEVEL_PERCENTILES = (2.0, 98.0)
ENERGY_LEVEL_LIMITS = None  # e.g. (0.0, 5.0e-3) to override per-plot limits

# Skip existing PDFs so rerunning the notebook is cheap and non-destructive.
SKIP_EXISTING = False
CONTINUE_ON_FAILURE = True

## Helpers

In [3]:
def _paper_spec(cfg, traj_idx=None):
    spec = replace(cfg.energy_landscape_spec)
    if traj_idx is not None:
        spec.traj_idx = int(traj_idx)
    spec.n_grid = int(PAPER_N_GRID)
    spec.n_levels = int(PAPER_N_LEVELS)
    spec.contour_style = "lines"
    spec.show_colorbar = bool(SHOW_COLORBAR)
    spec.show_path_points = False
    spec.show_start_end = False
    spec.path_linewidth = PATH_LINEWIDTH if SHOW_PATH else 0.0
    spec.contour_linewidth = float(PAPER_CONTOUR_LINEWIDTH)
    spec.title = None
    return spec


def _load_experiment(exp_dir):
    exp_dir = Path(exp_dir)
    with open(exp_dir / "config.json") as f:
        payload = json.load(f)

    cfg = _sweep_config_from_payload(payload)
    registry = build_architecture_registry()
    arch_name = payload.get("arch_name")
    if arch_name not in registry:
        raise ValueError(f"Unknown arch_name={arch_name!r} in {exp_dir / 'config.json'}")

    arch = registry[arch_name]
    params = make_model_params(cfg, arch)
    properties = _properties_from_payload(payload)
    base, aux = get_slinky(properties)

    train_file = _resolve_saved_data_path(payload, "train_file", str(exp_dir))
    valid_file = _resolve_saved_data_path(payload, "valid_file", str(exp_dir))
    train = Dataset.load(train_file, force_key=False)
    valid = Dataset.load(valid_file, force_key=False)

    return payload, cfg, arch, params, base, aux, train, valid


def _select_plot_data(cfg, train, valid, use_valid=None, traj_idx=None):
    use_valid = cfg.energy_snapshot_use_valid if use_valid is None else bool(use_valid)
    data = valid if use_valid else train
    traj_idx = cfg.energy_landscape_spec.traj_idx if traj_idx is None else int(traj_idx)

    idx_b = data.idx_b if data.idx_b.ndim == 1 else data.idx_b[traj_idx]
    xb = data.xb[traj_idx]
    lambdas = data.lambdas if data.lambdas.ndim == 1 else data.lambdas[traj_idx]
    return data, traj_idx, idx_b, xb, lambdas


def _resolve_traj_indices(cfg, train, valid, use_valid=None, traj_idxs=None):
    use_valid = cfg.energy_snapshot_use_valid if use_valid is None else bool(use_valid)
    data = valid if use_valid else train
    n_traj = int(data.qs.shape[0])

    if traj_idxs == "all":
        return list(range(n_traj))
    if traj_idxs is None:
        return [int(cfg.energy_landscape_spec.traj_idx)]
    if isinstance(traj_idxs, (int, np.integer)):
        return [int(traj_idxs)]

    resolved = [int(idx) for idx in traj_idxs]
    bad = [idx for idx in resolved if idx < 0 or idx >= n_traj]
    if bad:
        raise IndexError(f"Trajectory indices {bad} out of bounds for n_traj={n_traj}")
    return resolved


def _strain_path_for_model(model, base, aux, cfg, train, valid, spec, use_valid=None, traj_idx=None):
    _, selected_traj_idx, idx_b, xb, lambdas = _select_plot_data(
        cfg, train, valid, use_valid=use_valid, traj_idx=traj_idx
    )

    qs, auxs, del_strains = solve_with_strain_history(
        model=model,
        base=base,
        aux=aux,
        idx_b=idx_b,
        xb=xb,
        lambdas=lambdas,
        max_dlambda=cfg.max_dlambda,
        iters=cfg.iters,
        ls_steps=cfg.ls_steps,
    )

    resolved_triplet_idx = resolve_triplet_idx(np.asarray(del_strains).shape[-2], spec.triplet_idx)
    spec_local = replace(spec, triplet_idx=resolved_triplet_idx, traj_idx=0)
    path = extract_strain_path(del_strains, traj_idx=0, triplet_idx=resolved_triplet_idx)
    return path, spec_local, qs, auxs, del_strains, selected_traj_idx


def _energy_limits_from_values(values):
    finite = np.asarray(values)[np.isfinite(values)]
    if finite.size == 0:
        raise ValueError("Energy grid contains no finite values.")
    if ENERGY_LEVEL_LIMITS is not None:
        lo, hi = ENERGY_LEVEL_LIMITS
        lo, hi = float(lo), float(hi)
    elif LEVEL_PERCENTILES is None:
        lo, hi = float(np.min(finite)), float(np.max(finite))
    else:
        lo, hi = np.percentile(finite, LEVEL_PERCENTILES)
        lo, hi = float(lo), float(hi)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(finite)), float(np.max(finite))
    if hi <= lo:
        hi = lo + 1.0
    return lo, hi


def _contour_levels(values, n_levels):
    lo, hi = _energy_limits_from_values(values)
    return np.linspace(lo, hi, int(n_levels))


def _expand_limits(limits, scale=1.0, min_half_width=0.0):
    lo, hi = float(limits[0]), float(limits[1])
    center = 0.5 * (lo + hi)
    half_width = 0.5 * max(hi - lo, 0.0)
    half_width = max(half_width * float(scale), float(min_half_width))
    return center - half_width, center + half_width


def _plot_limits_from_path(x_path, y_path, spec):
    auto_x, auto_y = _auto_limits_from_path(x_path, y_path)
    xlim = MANUAL_XLIM if MANUAL_XLIM is not None else spec.xlim
    ylim = MANUAL_YLIM if MANUAL_YLIM is not None else spec.ylim
    if xlim is None:
        xlim = _expand_limits(auto_x, STRAIN_RANGE_SCALE, STRAIN_MIN_HALF_WIDTH)
    if ylim is None:
        ylim = _expand_limits(auto_y, STRAIN_RANGE_SCALE, STRAIN_MIN_HALF_WIDTH)
    return tuple(xlim), tuple(ylim)


def _plot_contours_only(model, path, spec, out_pdf, *, grid=None, levels=None):
    x_path = np.asarray(path[:, spec.strain_x_idx])
    y_path = np.asarray(path[:, spec.strain_y_idx])

    xlim, ylim = _plot_limits_from_path(x_path, y_path, spec)
    spec_eval = replace(spec, xlim=xlim, ylim=ylim)
    fixed_strains = _make_fixed_strains(path, spec_eval)
    if grid is None:
        X, Y, E = evaluate_energy_on_grid(model, spec_eval, fixed_strains)
    else:
        X, Y, E = grid
    if levels is None:
        levels = _contour_levels(E, spec_eval.n_levels)

    fig, ax = plt.subplots(figsize=FIGSIZE)
    contour = ax.contour(
        X,
        Y,
        E,
        levels=levels,
        cmap=PAPER_CMAP,
        linewidths=spec_eval.contour_linewidth,
    )
    if hasattr(contour, "collections"):
        for collection in contour.collections:
            collection.set_rasterized(False)

    if SHOW_CONTOUR_LABELS and len(contour.levels) > 0:
        label_levels = contour.levels[::max(1, int(CONTOUR_LABEL_EVERY))]
        ax.clabel(
            contour,
            levels=label_levels,
            inline=True,
            fontsize=CONTOUR_LABEL_FONT_SIZE,
            fmt=CONTOUR_LABEL_FORMAT,
        )

    if spec_eval.show_colorbar:
        cbar = fig.colorbar(contour, ax=ax, fraction=0.052, pad=0.025)
        cbar.set_label("Energy")
        cbar.ax.tick_params(labelsize=7, length=2)

    if SHOW_PATH:
        ax.plot(
            x_path,
            y_path,
            color=PATH_COLOR,
            linewidth=spec_eval.path_linewidth,
            solid_capstyle="round",
            zorder=3,
        )
        ax.scatter(
            [x_path[0]],
            [y_path[0]],
            s=START_MARKER_SIZE,
            marker="o",
            facecolors="white",
            edgecolors=PATH_COLOR,
            linewidths=0.8,
            zorder=4,
        )
        ax.scatter(
            [x_path[-1]],
            [y_path[-1]],
            s=END_MARKER_SIZE,
            marker="*",
            facecolors=PATH_COLOR,
            edgecolors=PATH_COLOR,
            linewidths=0.6,
            zorder=5,
        )

    ax.set_xlim(spec_eval.xlim)
    ax.set_ylim(spec_eval.ylim)
    ax.set_aspect("auto")

    if not SHOW_AXES:
        ax.set_axis_off()
        if not spec_eval.show_colorbar:
            fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")

    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_pdf, format="pdf", bbox_inches="tight", pad_inches=0, transparent=TRANSPARENT)
    plt.close(fig)

    return {
        "pdf": str(out_pdf),
        "xlim": tuple(spec_eval.xlim),
        "ylim": tuple(spec_eval.ylim),
        "levels": levels,
        "fixed_strains": np.asarray(fixed_strains),
    }


def _output_pdf_path(exp_dir, cfg, arch_name, tag, traj_idx):
    exp_dir = Path(exp_dir)
    filename = f"energy_landscape_{tag}_traj_{int(traj_idx):03d}_paper.pdf"
    if CENTRAL_OUTPUT_DIR is None:
        return exp_dir / cfg.energy_snapshot_dirname / filename

    central = Path(CENTRAL_OUTPUT_DIR)
    exp_slug = exp_dir.name
    return central / f"{arch_name}__{exp_slug}__{tag}__traj_{int(traj_idx):03d}_paper.pdf"


def _create_one_snapshot_pdf(model, base, aux, cfg, train, valid, spec, exp_dir, arch_name, tag, traj_idx):
    out_pdf = _output_pdf_path(exp_dir, cfg, arch_name, tag, traj_idx)
    result_key = f"{tag}_traj_{int(traj_idx):03d}"
    if SKIP_EXISTING and out_pdf.exists():
        return result_key, {"pdf": str(out_pdf), "skipped_existing": True}

    path, spec_local, *_ = _strain_path_for_model(
        model, base, aux, cfg, train, valid, spec, use_valid=USE_VALID, traj_idx=traj_idx
    )
    print(
        f"[{arch_name} {tag} traj {int(traj_idx)}] axes: "
        f"strain[{spec_local.strain_x_idx}] vs strain[{spec_local.strain_y_idx}]"
    )
    return result_key, _plot_contours_only(model, path, spec_local, out_pdf)


def create_paper_landscapes_for_experiment(exp_dir):
    payload, cfg, arch, params, base, aux, train, valid = _load_experiment(exp_dir)
    arch_name = payload.get("arch_name", Path(exp_dir).name)
    traj_indices = _resolve_traj_indices(cfg, train, valid, use_valid=USE_VALID, traj_idxs=TRAJ_IDXS)
    written = {}

    if INCLUDE_INITIAL:
        initial_model = arch.model_cls(params)
        for traj_idx in traj_indices:
            spec = _paper_spec(cfg, traj_idx=traj_idx)
            key, value = _create_one_snapshot_pdf(
                initial_model, base, aux, cfg, train, valid, spec, exp_dir, arch_name, "initial", traj_idx
            )
            written[key] = value

    if INCLUDE_FINAL:
        model_path = Path(exp_dir) / "model.eqx"
        if not model_path.is_file():
            written["final_missing_model"] = {"missing": str(model_path)}
            return arch_name, written

        model = eqx.tree_deserialise_leaves(model_path, arch.model_cls(params))
        for traj_idx in traj_indices:
            spec = _paper_spec(cfg, traj_idx=traj_idx)
            key, value = _create_one_snapshot_pdf(
                model, base, aux, cfg, train, valid, spec, exp_dir, arch_name, "final", traj_idx
            )
            written[key] = value

    return arch_name, written


def create_paper_landscapes_for_sweep(sweep_output_dir):
    exp_dirs = _find_saved_experiment_dirs(str(sweep_output_dir))
    if not exp_dirs:
        raise FileNotFoundError(f"No config.json files found under {sweep_output_dir!r}")

    n_models = sum((Path(exp_dir) / "model.eqx").is_file() for exp_dir in exp_dirs)
    print(f"Found {len(exp_dirs)} experiment configs; {n_models} have model.eqx artifacts.")

    results = {}
    for exp_dir in exp_dirs:
        try:
            arch_name, written = create_paper_landscapes_for_experiment(exp_dir)
            results[str(exp_dir)] = {"arch_name": arch_name, "outputs": written}
            print(f"[ok] {arch_name}: {written}")
        except Exception as exc:
            results[str(exp_dir)] = {"error": repr(exc)}
            print(f"[skip] {exp_dir}: {exc}")
            if not CONTINUE_ON_FAILURE:
                raise
    return results

## Generate PDFs

In [4]:
results = create_paper_landscapes_for_sweep(SWEEP_OUTPUT_DIR)
n_ok = sum("outputs" in item for item in results.values())
n_fail = sum("error" in item for item in results.values())
print(f"\nDone. Generated or found existing paper PDFs for {n_ok} experiments; {n_fail} failed/skipped.")

Found 14 experiment configs; 10 have model.eqx artifacts.


/var/folders/z0/3frv2l990hb5ryd49z8vm4w80000gn/T/ipykernel_39873/4181129550.py:31: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


[brazier_chol_stiffness_baseline initial traj 0] axes: strain[0] vs strain[3]
[brazier_chol_stiffness_baseline initial traj 1] axes: strain[0] vs strain[3]
[ok] brazier_chol_stiffness_baseline: {'initial_traj_000': {'pdf': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n11_tape_all_architectures_train_fail_on_nonconvergence_True/brazier_chol_stiffness_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.05__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05__mode_anisotropic/energy_landscapes/energy_landscape_initial_traj_000_paper.pdf', 'xlim': (-0.005856021946288153, 0.007462497843187926), 'ylim': (0.08948021377395726, 0.2573189550000131), 'levels': array([7.87681510e-05, 9.31758657e-05, 1.07583580e-04, 1.21991295e-04,
       1.36399010e-04, 1.50806725e-04, 1.65214439e-04, 1.79622154e-04,
       1.94029869e-04, 2.08437583e-04, 2.22845298e-04, 2.37253013e-04,
       2.51660727e-04, 2.66068442e-04, 2.80476157e-04, 2.948

/var/folders/z0/3frv2l990hb5ryd49z8vm4w80000gn/T/ipykernel_39873/4181129550.py:31: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


[brazier_chol_stiffness_icnn initial traj 0] axes: strain[0] vs strain[3]
[brazier_chol_stiffness_icnn initial traj 1] axes: strain[0] vs strain[3]
[brazier_chol_stiffness_icnn final traj 0] axes: strain[0] vs strain[3]
[brazier_chol_stiffness_icnn final traj 1] axes: strain[0] vs strain[3]
[ok] brazier_chol_stiffness_icnn: {'initial_traj_000': {'pdf': '/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/arch_sweep_outputs_n11_tape_all_architectures_train_fail_on_nonconvergence_True/brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.05__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05__mode_anisotropic/energy_landscapes/energy_landscape_initial_traj_000_paper.pdf', 'xlim': (-0.1283915342972661, 0.003613760480616096), 'ylim': (-0.09446601625513268, 0.37090985114142255), 'levels': array([0.00035593, 0.00150983, 0.00266374, 0.00381764, 0.00497155,
       0.00612545, 0.00727936, 0.00843327, 0.00958717, 0.01074108,
       0.01